# 05 常態分布與模擬（講義 06a / 06b）

In [ ]:
import math
import random
import statistics

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
from scipy import stats

import statkit as sk
sk.setup()  # 字體大小 16、中文字型

## 常態分布 ND(μ, σ)

⚠ scipy 的 `scale` 是**標準差** σ，不是變異數。

互動式的 `sk.explore()` 在 VS Code 裡需要網路才顯示得出輸入框，**考試斷網時請在 JupyterLab 用**；VS Code 裡直接寫 `X.P(...)`、`X.quantile(...)`、`sk.plot_dist(...)` 就好。

In [ ]:
mean, stdev = 80, 10
print(f"f(70) = {stats.norm.pdf(x=70, loc=mean, scale=stdev):0.4f}")
print(f"F(70) = {stats.norm.cdf(x=70, loc=mean, scale=stdev):0.4f}")
print(f"x for F(x) = 0.8: {stats.norm.ppf(q=0.8, loc=mean, scale=stdev):0.4f}")

## 直接算 vs 標準化（z）— 講義 06a 例題

In [ ]:
mean, stdev = 6.5, 2
print(f'Pr(X >= 10) = 1 - F(10) = {1 - stats.norm.cdf(x=10, loc=mean, scale=stdev):0.4f}')
print(f'Pr(X <= 0)  = F(0)      = {stats.norm.cdf(x=0, loc=mean, scale=stdev):0.4f}')
print(f'Using SND: Pr(Z >= {(10 - mean) / stdev}) = {1 - stats.norm.cdf(x=(10 - mean) / stdev):0.4f}')
print(f'x for Pr(X <= x) = 0.6: {stats.norm.ppf(q=0.6, loc=mean, scale=stdev):0.4f}')
print(f'               via z : {stats.norm.ppf(q=0.6) * stdev + mean:0.4f}')

In [ ]:
X = sk.Normal(6.5, 2)
X.P('X >= 10')
X.quantile(0.6)
X.quantile(0.05, tail='right')     # Pr(X ≥ x) = 0.05
sk.plot_dist(X, '4 <= X <= 8')
plt.show()

## 右尾反查、包含中間 p% 的區間

- 「70% 的人比 x 高」：$\Pr(X > x) = 0.7$ → `ppf(1 - 0.7)`（或 `isf(0.7)`）
- 「中間 95% 落在 $[L, U]$」：兩邊各留 2.5% → `ppf(0.025)`、`ppf(0.975)`；經驗法則的 $\mu \pm 2\sigma$ 是近似值

In [ ]:
mean, stdev = 100, 15
print('Pr(X > x) = 0.7 → x =', round(stats.norm.ppf(q=1 - 0.7, loc=mean, scale=stdev), 4))
print('            用 isf：', round(stats.norm.isf(0.7, loc=mean, scale=stdev), 4))
for p in [0.68, 0.95]:
    lower = stats.norm.ppf((1 - p) / 2, loc=mean, scale=stdev)   # 左尾 (1 − p)/2
    upper = stats.norm.ppf((1 + p) / 2, loc=mean, scale=stdev)   # 右尾 (1 − p)/2
    print(f'中間 {p:.0%}：[{lower:.4f}, {upper:.4f}]')

In [ ]:
sk.middle_interval(sk.Normal(100, 15), 0.95)

## 常用 z 值

In [ ]:
for q in [0.90, 0.95, 0.975, 0.99, 0.995]:
    print(f'z_{1 - q:.3f} = {stats.norm.ppf(q):.4f}')

## 常態的線性函數與線性組合（講義 06a 定理 1、2）

- $X \sim \text{ND}(\mu, \sigma)$ → $aX + b \sim \text{ND}(a\mu + b, |a|\sigma)$（講義寫 $a\sigma$；$a < 0$ 時要取絕對值）
- $X_i$ 互相獨立 → $\sum_i a_i X_i \sim \text{ND}\left(\sum_i a_i\mu_i, \sqrt{\sum_i a_i^2\sigma_i^2}\right)$

⚠ **變異數相加，不是標準差相加**。兩個獨立、都是 ND(30, 5) 的時間加總，標準差是 $\sqrt{5^2 + 5^2} \approx 7.07$；**同一個**變數乘 2（$2X$）標準差才是 10。

In [ ]:
# X ~ ND(50, 6)、Y ~ ND(20, 2) 獨立，W = 3X − 2Y + 10
mu = 3 * 50 - 2 * 20 + 10
sigma = math.sqrt(3 ** 2 * 6 ** 2 + (-2) ** 2 * 2 ** 2)     # 係數平方 × 變異數，再相加
print('W ~ ND(', mu, ',', round(sigma, 4), ')')
print('Pr(W > 130) =', round(1 - stats.norm.cdf(x=130, loc=mu, scale=sigma), 4))

print('X1 + X2 的標準差：', round(math.sqrt(5 ** 2 + 5 ** 2), 4), '；2X 的標準差：', 2 * 5)

statkit：常態可以直接寫運算，結果還是常態；`.explain()` 印出 μ、σ 的算式（同一個變數出現兩次會自動合併成 2X）。

In [ ]:
X, Y = sk.Normal(50, 6), sk.Normal(20, 2)
W = 3 * X - 2 * Y + 10
W.explain()
W.P('X > 130')      # 條件裡的 X 代表 W 本身

## 二項的常態近似（連續校正）

$X \sim Bi(n, p) \approx ND(np, \sqrt{np(1-p)})$，$\Pr(a \le X \le b) \approx \Pr(a - 0.5 \le Y \le b + 0.5)$

In [ ]:
trial_cnt, prob_success = 50, 0.5
prob_bi = stats.binom.cdf(k=30, n=trial_cnt, p=prob_success) - stats.binom.cdf(k=24, n=trial_cnt, p=prob_success)
mean = trial_cnt * prob_success
stdev = (trial_cnt * prob_success * (1 - prob_success)) ** (1 / 2)
prob_without = stats.norm.cdf(x=30, loc=mean, scale=stdev) - stats.norm.cdf(x=25, loc=mean, scale=stdev)
prob_with = stats.norm.cdf(x=30.5, loc=mean, scale=stdev) - stats.norm.cdf(x=24.5, loc=mean, scale=stdev)
print(f'Pr(25 <= X <= 30)     = {prob_bi:0.4f}')
print(f'Pr(25 <= Y <= 30)     = {prob_without:0.4f}')
print(f'Pr(24.5 <= Y <= 30.5) = {prob_with:0.4f}')

In [ ]:
sk.normal_approx(50, 0.5, '25 <= X <= 30')
plt.show()

## 抽樣平均數的分布（中央極限定理）

$E[\bar{X}] = \mu$，$\operatorname{Var}(\bar{X}) = \sigma^2 / n$

In [ ]:
def simulate_sample_mean(mean, stdev, exp_cnt, sample_size):
    population = list(stats.norm.rvs(loc=mean, scale=stdev, size=1000, random_state=1))
    sample_means = [statistics.mean(random.sample(population, sample_size)) for _ in range(exp_cnt)]
    print(f'n = {sample_size}: mean of means {statistics.mean(sample_means):0.4f}, '
          f'variance {statistics.pvariance(sample_means):0.4f} (theory {stdev ** 2 / sample_size})')

random.seed(115)
for n in [4, 9, 16]:
    simulate_sample_mean(80, 4, 500, n)

In [ ]:
means = sk.sample_means(sk.Expon(1), n=30, reps=2000, seed=1)   # 母體不是常態也會趨近常態
plt.show()

## 樣本平均數：怎麼算機率（講義 06a 例題）

母體為常態時 $\bar{X} \sim \text{ND}\left(\mu, \frac{\sigma}{\sqrt{n}}\right)$（不是常態時，$n$ 夠大也近似成立）。講義例題：從 ND(80, 10) 抽 4 個 → $\bar{X} \sim \text{ND}(80, 5)$。

「樣本平均數**偏離** μ 超過 d」要算兩邊：$\Pr(|\bar{X} - \mu| > d) = 2\Pr(\bar{X} > \mu + d)$。

In [ ]:
mu, sigma, n = 80, 10, 4
se = sigma / math.sqrt(n)                    # 樣本平均數的標準差 σ/√n = 5
print('Pr(X̄ > 82) =', round(1 - stats.norm.cdf(x=82, loc=mu, scale=se), 4))      # 講義 ≈ 0.345
print('Pr(X̄ < 76) =', round(stats.norm.cdf(x=76, loc=mu, scale=se), 4))          # 講義 ≈ 0.212
print('Pr(|X̄ − 80| > 5) =', round(2 * stats.norm.sf(85, loc=mu, scale=se), 4))   # 兩邊 ×2

In [ ]:
Xbar = sk.Normal(80, 10).sample_mean(4)
Xbar.P('X > 82')
Xbar.P('X <= 75 or X >= 85')     # 兩邊的尾巴

**講義 Pepsi 例題**：抽 500 位學生，平均每週 1.316 罐、樣本標準差 1.1147。如果真正的 μ 只有 1.189（市占率 37% 的門檻），看到 $\bar{X} \ge 1.316$ 的機率：

In [ ]:
se = 1.1147 / math.sqrt(500)
print('σ/√n =', round(se, 4))                                                   # ≈ 0.049
print('Pr(X̄ ≥ 1.316) =', round(1 - stats.norm.cdf(x=1.316, loc=1.189, scale=0.049), 4))   # 講義用 0.049 → 0.0048
# 機率非常小 → μ 不太可能只有 1.189 → 市占率確實低於 37%

## 需要多大的樣本

要讓 $\Pr(|\bar{X} - \mu| \le d) \ge$ prob：$\frac{d}{\sigma/\sqrt{n}} \ge z$ → $n \ge \left(\frac{z\sigma}{d}\right)^2$，$z$ = `ppf((1 + prob) / 2)`，**無條件進位**。「偏離超過 d 的機率不超過 1%」就是 prob = 0.99。

In [ ]:
sigma, d, prob = 10, 2, 0.95
z = stats.norm.ppf((1 + prob) / 2)
n = math.ceil((z * sigma / d) ** 2)
print('z =', round(z, 4), '→ n =', n)

In [ ]:
sk.sample_size(sigma=10, margin=2, prob=0.95)

## 疊圖：不同樣本大小的樣本平均數（講義 06b）

n 越大，樣本平均數越集中在 μ 附近（變異數 σ²/n 變小）。同一張圖疊三組：`alpha` 半透明、`label` + `legend` 區分，再疊上理論的常態曲線。

In [ ]:
rng = np.random.default_rng(115)
population_mean, population_sd = 80, 8
xs = np.linspace(68, 92, 300)
for n, color in [(4, 'tab:blue'), (9, 'tab:orange'), (16, 'tab:green')]:
    means = rng.normal(population_mean, population_sd, size=(500, n)).mean(axis=1)   # 500 組樣本平均數
    plt.hist(means, bins=range(68, 93), density=True, alpha=0.35, color=color, label=f'n = {n}')
    plt.plot(xs, stats.norm.pdf(xs, loc=population_mean, scale=population_sd / math.sqrt(n)), color=color)
plt.xlabel('Sample mean')
plt.ylabel('Density')
plt.title('Sample means with different sample sizes')
plt.legend()
plt.show()

## Bessel 校正：為什麼樣本變異數除以 n − 1（講義 06b）

In [ ]:
rng = np.random.default_rng(1)
samples = rng.normal(0, 2, size=(20000, 5))      # 母體變異數 = 4
print('除以 n    :', samples.var(axis=1, ddof=0).mean().round(3))
print('除以 n - 1:', samples.var(axis=1, ddof=1).mean().round(3))

## 隨機變數的柴比雪夫定理、大數法則（講義 06b）

- **柴比雪夫**：任何分布 $\Pr(|X - \mu| \ge k\sigma) \le \frac{1}{k^2}$（$k > 1$）。不知道分布時也能給機率的上界；知道是常態時，實際機率小很多。
- **大數法則**：樣本越大，$\bar{X}_n$ 越接近 μ。

In [ ]:
k = 2
print('柴比雪夫上界：', 1 / k ** 2)
print('如果是常態：  ', round(2 * stats.norm.sf(k), 4))     # Pr(|Z| ≥ 2)

rng = np.random.default_rng(1)
rolls = rng.integers(1, 7, size=5000)                     # 擲骰子，μ = 3.5
running_mean = np.cumsum(rolls) / np.arange(1, len(rolls) + 1)
plt.plot(running_mean, color='black')
plt.axhline(3.5, color='red', linestyle='--', label='μ = 3.5')
plt.xlabel('n')
plt.ylabel('Sample mean')
plt.title('Law of large numbers')
plt.legend()
plt.show()

## 蒙地卡羅：估 π、三門問題

In [ ]:
random.seed(115)
point_cnt = 2000
inside = sum(((random.uniform(0, 2) - 1) ** 2 + (random.uniform(0, 2) - 1) ** 2) <= 1 for _ in range(point_cnt))
print('pi ≈', 4 * inside / point_cnt)

sk.estimate_pi(2000, seed=115)
sk.monty_hall(10000, seed=115)

## 亂數小提醒
- 要讓結果可重現：`random.seed(...)`、`np.random.default_rng(seed)`、scipy 加 `random_state=...`
- `stats.norm.rvs(loc, scale, size)`、`random.sample(population, k)`（不放回）、`random.choices(population, k=...)`（放回）